# Exploration-Exploitation Challenge {#sec-exploration}

Cazé and Van der Meer (2013) showed that optimism/pessimism have an impact on exploration on a two-armed bandit task; importantly, this impact is modulated by resource scarcity [@caze2013]. Here, we see whether we can reproduce this effect. First, we implement optimism/pessimism with the three biasing methods defined in @sec-biasing-methods . We find that the weights method closely approximates previous results on a two-armed bandit task. Then, we replicate Cazé and Van der Meer's findings, showing that optimism is beneficial in scarce environments, while pessimism is advantageous in the abundant version of the task.

First, we import what we need:

In [ ]:
# general imports

import numpy as np
from scipy.stats import norm

import jax.numpy as jnp
from jax import jit
from functools import partial

import matplotlib.pyplot as plt
import matplotlib as mpl

import pandas as pd
import seaborn as sns

In [ ]:
# pyCRLD imports

# Agents
from pyCRLD.Agents.Base import abase
from pyCRLD.Agents.StrategySARSA import stratSARSA

# Environment
from pyDDRL.Environments.TwoArmedBandit import TwoArmedBandit

# Utilities
from pyCRLD.Utils.Helpers import *
from pyDDRL.Utils.Trajectories1D import trajectories_1D 
from pyDDRL.Utils.PhasePlot import phaseplot, _data_to_plot, _strategies, _dXisa_s
from pyDDRL.Utils.PlottingTools import *

In [ ]:
# DDRL imports

from pyDDRL.Agents.DDRLBase import DDRLBase
from pyDDRL.Agents.DDRLSarsa import DDRLSarsa

## Comparison between the three DDRL biasing methods and asymmetric RL

Asymmetric RL was previously used to model optimism and pessimism in simple RL algorithms. It postulates a pair of asymmetric learning rates: a learning rate $\alpha^{+}$ that updates positive prediction errors, and a rate $\alpha^{-}$ that updates negative prediction errors. The bias strength $b$ is defined by the learning-rates ratio: 

$$ b = \frac{\alpha^{+}}{\alpha^{-}} $$

Cazé and Van der Meer (2013) [@caze2013] have shown that, for an agent with bias strength $b$, the Q-value at steady-state of arm $a_{i}$ is:

$$ Q^{a_i}_{\infty} = \frac{p_{a_i} \cdot b - (1 - p_{a_i})}{p_{a_i} \cdot b + (1 - p_{a_i})} $$

Below, we compare the final strategies obtained with the three DDRL biasing methods (quantile, expectile, weights) against the final strategies obtained with asymmetric RL.

Now, we create lists of bias strengths, quantiles, expectiles and weights. 

The list of bias strengths is:

In [ ]:
b_pes = np.linspace(0.1, 1., 10)  # pessimistic values
b_opt = np.linspace(1., 10., 10)  # optimistic values
b_ratios = np.zeros((19))
b_ratios[:9] = b_pes[:9]
b_ratios[9:] = b_opt
b_ratios

array([ 0.1,  0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8,  0.9,  1. ,  2. ,
        3. ,  4. ,  5. ,  6. ,  7. ,  8. ,  9. , 10. ])

Rowland *et al.* (2021) showed that, in stateless environments, expectile-based distributional RL is equivalent with asymmetric RL, when $\tau = \frac{b}{b+1}$ [@rowland2021temporal].

We want to investigate whether this is the case on a two-armed bandit task, so we define the expectile list as:

In [ ]:
e_list = b_ratios / (b_ratios + 1)
e_list

array([0.09090909, 0.16666667, 0.23076923, 0.28571429, 0.33333333,
       0.375     , 0.41176471, 0.44444444, 0.47368421, 0.5       ,
       0.66666667, 0.75      , 0.8       , 0.83333333, 0.85714286,
       0.875     , 0.88888889, 0.9       , 0.90909091])

Next, we define a quantile list with $\tau$ spanning the range of $[0,1]$:

In [ ]:
q_list = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30,
          0.35, 0.40, 0.45, 0.50, 0.55, 0.60,
          0.65, 0.70, 0.75, 0.80, 0.85, 0.90,
          0.95]

Finally, we use a similar list for the upper weights $w_U$ as the one we used for the bias strengths $b$:

In [ ]:
w_pes = np.linspace(0.1, 1., 10)
w_opt = np.linspace(1., 10., 10)
w_list = np.zeros((19))
w_list[:9] = w_pes[:9]
w_list[9:] = w_opt
w_list

array([ 0.1,  0.2,  0.3,  0.4,  0.5,  0.6,  0.7,  0.8,  0.9,  1. ,  2. ,
        3. ,  4. ,  5. ,  6. ,  7. ,  8. ,  9. , 10. ])

We can now compute and plot our comparisons on a poor two-armed bandit task. Here, we use a distributional SARSA agent with reward sensitivity $\sigma = 0.0001$, since we established in @sec-calibration that this value of $\sigma$ minimizes the error on a two-armed bandit task.

In [ ]:
# Compute comparisons in poor env
b_xinf, q_xinf, e_xinf, w_xinf = compute_comparisons(0.3, 0.1, b_ratios, q_list, e_list, 
                                                     w_list, gamma=0, beta=4, Rmin=-1, 
                                                     Rmax=1, nrb=29, discsig=0.0001, lr=0.1)

In [ ]:
#| label: fig-comparison-poor
#| fig-cap: "Final strategy of an agent playing a poor two-armed bandit task as a function of bias strength. The quantile, expectile, and weight methods are compared with asymmetric RL."

# Plot comparisons in poor env
plot_comparisons(b_ratios, q_list, e_list, w_list, b_xinf, 
                 q_xinf, e_xinf, w_xinf, scarcity="poor",
                 save=True, savename='../figs/TwoArmedMethodComparisonPoor.png')

plt.show()

<Figure size 5130x1350 with 6 Axes>

And now, in a rich environment:

In [ ]:
# Compute comparisons in a rich environment
b_xinf, q_xinf, e_xinf, w_xinf = compute_comparisons(0.9, 0.7, b_ratios, q_list, e_list, 
                                                     w_list, gamma=0, beta=4, Rmin=-1, 
                                                     Rmax=1, nrb=29, discsig=0.0001, lr=0.1)

In [ ]:
#| label: fig-comparison-rich
#| fig-cap: "Final strategy of an agent playing a rich two-armed bandit task as a function of bias strength. The quantile, expectile, and weight methods are compared with asymmetric RL."

# Plot comparisons in a rich environment
plot_comparisons(b_ratios, q_list, e_list, w_list, b_xinf, 
                 q_xinf, e_xinf, w_xinf, scarcity="rich",
                 save=True, savename='../figs/TwoArmedMethodComparisonRich.png')

plt.show()

<Figure size 5130x1350 with 6 Axes>

The figures show a close approximation of asymmetric RL by weights-DDRL. The equivalence between the two methods can be formally proven. 

## Proof: Equivalence between weights-DDRL and asymmetric RL in a two-armed bandit task

We consider a multi-armed bandit with arms $a_i$, $i \in N$; for each $i$, $a_i$ yields a reward, $+1$, with probability $p_i$, and a punishment, $-1$, with probability $1-p_i$. 

An asymmetric RL agent is equipped with two different learning rates: $\alpha^{+}$ updates positive prediction errors, and $\alpha^{-}$ updates negative prediction errors. The agent's bias strength, $b$, is defined as the learning-rate ratio: $b = \frac{\alpha^{+}}{\alpha^{-}}$. Cazé and van der Meer (2013) have shown that, for such an agent, the Q-value at steady state for arm $a_{i}$ can be expressed as follows:

\begin{equation}
    Q^{a_i}_{\infty} = \frac{bp_i - (1-p_i)}{bp_i + (1-p_i)}
\end{equation}

Let us consider a weights-DDRL agent with number of neurons $n$, discretization sigma $\sigma$, discretization grid ${x_j, j \in [1, n]}$ with width $\Delta x$, and weight vector $\boldsymbol{w}$ with elements $w_j$, $j \in [1, n]$. For each arm $a_i$, this agent computes a Q-value distribution $Q^{a_i, v}$, which quantifies the probability that the Q-value of arm $a_i$ equals a value $v$. Since the two-armed bandit is a stateless environment, the Q-value distribution $Q^{a_i, v}$ equals the reward distribution $R^{a_i, r}$, which quantifies the probability that arm $a_{i}$ yields a reward $r$. 

Therefore, in the limit of an infinitely small discretization sigma $\sigma \rightarrow 0$, the Q-value distribution for arm $a_i$ can be described as follows:

* $Q^{a_i, v} = p_i$ when $v = 1$
* $Q^{a_i, v} = 1 - p_i$ when $v = -1$
* $Q^{a_i, v} = 0$ for all other values of $v$.

This is assuming that the agent is neutral -- that is, all elements $w_j$ of vector $\boldsymbol{w}$ have value $1$. Now, let us consider an optimistic/pessimistic agent. For such an agent, $w_j = 1$ if $x_j < \bar Q^{a_i, v}$, and $w_j = w_U$ if $x_j \geq \bar Q^{a_i, v}$, where $\bar Q^{a_i, v}$ denotes the mean of the Q-value distribution. 

The mean of the Q-value distribution is $\bar Q^{a_i, v} = 2p_i - 1$, hence $\bar Q^{a_i, v} \in [-1, 1]$. Excluding the extreme case in which there is an $i$, such that $p_i = 1$, we can formulate the weighted Q-value distribution $\tilde Q^{a_i, v}$:

* $\tilde Q^{a_i, v} = w_U p_i$ when $v = 1$
* $\tilde Q^{a_i, v} = 1 - p_i$ when $v = -1$
* $\tilde Q^{a_i, v} = 0$ for all other values of $v$.

We then normalize this distribution so that $\int_{x_1 - \frac{\Delta x}{2}}^{x_n+\frac{\Delta x}{2}}\tilde Q^{a_i, v} dv = 1$. To do this, we divide the distribution by its integral, hence:

* $\tilde Q^{a_i, v} = \frac{w_U p_i}{w_U p_i + (1-p_i)}$ when $v = 1$
* $\tilde Q^{a_i, v} = \frac{1 - p_i}{w_U p_i + (1-p_i)}$ when $v = -1$
* $\tilde Q^{a_i, v} = 0$ for all other values of $v$.

The mean of this new distribution is:
$$ \overset{\simeq} Q^{a_i, v} = \frac{w_U p_i}{w_U p_i + (1-p_i)} - \frac{1 - p_i}{w_U p_i + (1-p_i)} \\
    = \frac{w_U p_i - (1 - p_i)}{w_U p_i + (1-p_i)} $$

Thus, on a multi-armed bandit task, an asymmetric-RL agent with bias strength $b$ learns the same Q-values at steady state as a weights-DDRL agent with upper weight value $w_U = b$.

## Influence of optimism/pessimism on performance 

Here, we investigate how optimism/pessimism impact performance on a two-armed bandit task, when resources are scarce and when they are abundant.

### In a poor environment

In a "poor" two-armed bandit task, rewards are scarce. We thus define a two-armed bandit task with $p_1 = 0.3$, and $p_2 = 0.1$.

In [ ]:
# Poor environment

xs = np.array([[-1, 1],
               [-1, 1]])
ps = np.array([[0.7, 0.3],
               [0.9, 0.1]])
env = TwoArmedBandit(disttype='discrete', xs=xs, ps=ps)

Now, we can plot the phase plot, strategy over time, and reward over time, of an agent which performs this task. We compare several levels of optimism and pessimism. These levels are defined by the upper weights $w_U$ - that is, the weights the agent applies to the upper part of the distribution.

In [ ]:
# List of weight values to compare
weights = np.array([0.1, 0.2, 0.4, 0.8, 1., 2., 4., 8., 10.])

# i,s,a to plot along the x-axis
x = ([0], [0], [0]) 

# Range and resolution of X
res = np.linspace(0, 1., 21) 
res[0] = 0.0001
res[-1] = 0.9999

In @sec-calibration , we found that the discretization sigma yielding the lowest error for a neutral agent on a two-armed bandit task is $\sigma=0.0001$. We therefore use it to compute our results.

In [ ]:
#| label: fig-phaseplot-poor
#| fig-cap: "Phase plot, strategy plot and reward plot of a distributional agent playing a poor two-armed bandit task. Different levels of optimism/pessimism (quantified by the upper weight value $w_U$) are compared."

compare_levels(env, weights, gamma=0., beta=4., Rmin=-1, Rmax=1, 
               xinds=x, res=res, nrb=29, discsig=0.0001, traj_len=100, 
               cmap='coolwarm', save=True, 
               savename='../figs/TwoArmedPoorPhasePlot.png')

plt.show()

<Figure size 5130x1140 with 3 Axes>

### In a rich environment

In a "rich" two-armed bandit task, rewards are abundant. We thus define a two-armed bandit task with $p_1 = 0.9$, and $p_2 = 0.7$.

In [ ]:
# Rich environment

xs = np.array([[-1, 1],
               [-1, 1]])
ps = np.array([[0.1, 0.9],
               [0.3, 0.7]])
env = TwoArmedBandit(disttype='discrete', xs=xs, ps=ps)

Now, we can plot the phase plot, strategy over time, and reward over time, of an agent performing the rich task. We compare the same levels of optimism and pessimism as above.

In [ ]:
#| label: fig-phaseplot-rich
#| fig-cap: "Phase plot, strategy plot and reward plot of a distributional agent playing a rich two-armed bandit task. Different levels of optimism/pessimism (quantified by the upper weight value $w_U$) are compared."

compare_levels(env, weights, gamma=0., beta=4., Rmin=-1, Rmax=1, 
               xinds=x, res=res, nrb=29, discsig=0.0001, traj_len=100, 
               cmap='coolwarm', save=True, 
               savename='../figs/TwoArmedRichPhasePlot.png')

plt.show()

<Figure size 5130x1140 with 3 Axes>